# Data Collection

In [ ]:
import os
import pandas as pd

NFL_YEARS = [str(year) for year in range(2010, 2026)]

DATA_FILEPATH = r"Data"

def load_data(NFL_YEAR):
    path = os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv")
    NFL = pd.read_csv(path)
    return NFL

for year in NFL_YEARS:
    df = load_data(year)
    print(f"{year} loaded — {len(df)} teams")

display(df.tail())

# Logistic Regression Model

## Greg Yonan

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Column indices
TEAM = 0; WINS = 1; LOSSES = 2; TIES = 3; WIN_PERCENTAGE = 4
POINTS_FOR = 5; POINTS_AGAINST = 6; POINTS_DIFFERENTIAL = 7
MARGIN_OF_VICTORY = 8; STRENGTH_OF_SCHEDULE = 9
SIMPLE_RATING_SYSTEM = 10; OFFENSIVE_SRS = 11; DEFENSIVE_SRS = 12

NFL_TEAMS = ['Arizona Cardinals', 'Atlanta Falcons', 'Baltimore Ravens', 'Buffalo Bills',
             'Carolina Panthers', 'Chicago Bears', 'Cincinnati Bengals', 'Cleveland Browns',
             'Dallas Cowboys', 'Denver Broncos', 'Detroit Lions', 'Green Bay Packers',
             'Houston Texans', 'Indianapolis Colts', 'Jacksonville Jaguars', 'Kansas City Chiefs',
             'Las Vegas Raiders', 'Los Angeles Chargers', 'Los Angeles Rams', 'Miami Dolphins',
             'Minnesota Vikings', 'New England Patriots', 'New Orleans Saints', 'New York Giants',
             'New York Jets', 'Philadelphia Eagles', 'Pittsburgh Steelers', 'San Francisco 49ers',
             'Seattle Seahawks', 'Tampa Bay Buccaneers', 'Tennessee Titans', 'Washington Commanders']
NFL_YEARS = [str(year) for year in range(2010, 2026)]

DATA_FILEPATH = "Data"
SCHEDULE_FILEPATH = "Schedule"
MODEL_FILEPATH = "Models/Logistic Regression"
PREDICTIONS_FILEPATH = "Predictions/Logistic Regression"

In [ ]:
def read_data(NFL_YEAR, team_name):
    team_data = []
    with open(os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv"), "r") as file:
        data = pd.read_csv(file)
        for index, row in data.iterrows():
            if row.iloc[TEAM] == team_name:
                team_data.append([
                    row.iloc[WINS], row.iloc[LOSSES], row.iloc[TIES], row.iloc[WIN_PERCENTAGE],
                    row.iloc[POINTS_FOR], row.iloc[POINTS_AGAINST], row.iloc[POINTS_DIFFERENTIAL],
                    row.iloc[MARGIN_OF_VICTORY], row.iloc[STRENGTH_OF_SCHEDULE],
                    row.iloc[SIMPLE_RATING_SYSTEM], row.iloc[OFFENSIVE_SRS], row.iloc[DEFENSIVE_SRS]
                ])
    return pd.DataFrame(team_data, columns=['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'PD', 'MoV', 'SoS', 'SRS', 'OSRS', 'DSRS'])

In [ ]:
def train_model(NFL_YEAR):
    all_data = [read_data(NFL_YEAR, team) for team in NFL_TEAMS]
    combined_data = pd.concat([d for d in all_data if d is not None and len(d) > 0], ignore_index=True)

    for col in combined_data.columns:
        combined_data[col] = pd.to_numeric(combined_data[col], errors='coerce')
    combined_data = combined_data.dropna()

    features = combined_data.drop(columns=['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'MoV'])
    target = combined_data['W-L%']

    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model

In [ ]:
def save_model(NFL_YEAR, model):
    os.makedirs(MODEL_FILEPATH, exist_ok=True)
    with open(os.path.join(MODEL_FILEPATH, f"linear_regression_model_{NFL_YEAR}.pkl"), "wb") as file:
        pickle.dump(model, file)

In [ ]:
def make_predictions(NFL_YEAR):
    model_path = os.path.join(MODEL_FILEPATH, f"linear_regression_model_{NFL_YEAR}.pkl")
    with open(model_path, "rb") as file:
        model = pickle.load(file)

    schedule_df = pd.read_csv(os.path.join(SCHEDULE_FILEPATH, f"{NFL_YEAR}.csv"))
    if schedule_df.empty:
        print(f"No schedule found for {NFL_YEAR}")
        return

    team_wins  = {team: 0 for team in NFL_TEAMS}
    team_games = {team: 0 for team in NFL_TEAMS}

    try:
        season_data = pd.read_csv(os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv"))
        team_stats = {row.iloc[TEAM]: read_data(NFL_YEAR, row.iloc[TEAM]) for _, row in season_data.iterrows()}
        team_stats = {k: v for k, v in team_stats.items() if v is not None and len(v) > 0}
    except FileNotFoundError:
        print(f"Data file for {NFL_YEAR} not found")
        return

    drop_cols = ['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'MoV']
    for _, row in schedule_df.iterrows():
        winner, loser = row['Winner/tie'], row['Loser/tie']
        if winner not in team_stats or loser not in team_stats:
            continue
        winner_pred = model.predict(team_stats[winner].drop(columns=drop_cols))[0]
        loser_pred  = model.predict(team_stats[loser].drop(columns=drop_cols))[0]
        predicted_winner = winner if winner_pred > loser_pred else loser
        predicted_loser  = loser  if winner_pred > loser_pred else winner
        team_wins[predicted_winner] += 1
        team_games[predicted_winner] += 1
        team_games[predicted_loser]  += 1

    predictions = [
        [team,
         team_stats[team]['W-L%'].iloc[0] if team in team_stats else 0.0,
         team_wins[team] / team_games[team] if team_games[team] > 0 else 0.0]
        for team in NFL_TEAMS
    ]
    predictions_df = pd.DataFrame(predictions, columns=['Team', 'Actual W-L%', 'Predicted W-L%'])
    os.makedirs(PREDICTIONS_FILEPATH, exist_ok=True)
    predictions_df.to_csv(os.path.join(PREDICTIONS_FILEPATH, f"predictions_{NFL_YEAR}.csv"), index=False)
    return predictions_df

In [ ]:
for year in NFL_YEARS:
    print(f"Training model for {year}...")
    model = train_model(year)
    save_model(year, model)
    print(f"  Model saved.")
    preds = make_predictions(year)
    print(f"  Predictions made.\n")
    display(preds)  # shows the DataFrame inline in the notebook

# Neural Network Model

## Dylan Shaffer

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

TEAM = 0; WINS = 1; LOSSES = 2; TIES = 3; WIN_PERCENTAGE = 4
POINTS_FOR = 5; POINTS_AGAINST = 6; POINTS_DIFFERENTIAL = 7
MARGIN_OF_VICTORY = 8; STRENGTH_OF_SCHEDULE = 9
SIMPLE_RATING_SYSTEM = 10; OFFENSIVE_SRS = 11; DEFENSIVE_SRS = 12

DATA_FILEPATH = "Data"
MODEL_FILEPATH = "Models/Neural Network"
PREDICTIONS_FILEPATH = "Predictions/Neural Network"

DROP_COLS = ['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'MoV']

In [ ]:
def read_data(NFL_YEAR, team_name):
    team_data = []
    with open(os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv"), "r") as file:
        data = pd.read_csv(file)
        for index, row in data.iterrows():
            if row.iloc[TEAM] == team_name:
                team_data.append([
                    row.iloc[WINS], row.iloc[LOSSES], row.iloc[TIES], row.iloc[WIN_PERCENTAGE],
                    row.iloc[POINTS_FOR], row.iloc[POINTS_AGAINST], row.iloc[POINTS_DIFFERENTIAL],
                    row.iloc[MARGIN_OF_VICTORY], row.iloc[STRENGTH_OF_SCHEDULE],
                    row.iloc[SIMPLE_RATING_SYSTEM], row.iloc[OFFENSIVE_SRS], row.iloc[DEFENSIVE_SRS]
                ])
    return pd.DataFrame(team_data, columns=['W', 'L', 'T', 'W-L%', 'PF', 'PA', 'PD', 'MoV', 'SoS', 'SRS', 'OSRS', 'DSRS'])

In [ ]:
def build_model(input_dim):
    model = keras.Sequential([
        layers.Dense(64, activation="relu", input_shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(1)  # single output: W-L%
    ])
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model

In [ ]:
def train_model(NFL_YEAR):
    all_data = [read_data(NFL_YEAR, team) for team in NFL_TEAMS]
    combined_data = pd.concat([d for d in all_data if d is not None and len(d) > 0], ignore_index=True)

    for col in combined_data.columns:
        combined_data[col] = pd.to_numeric(combined_data[col], errors='coerce')
    combined_data = combined_data.dropna()

    features = combined_data.drop(columns=DROP_COLS)
    target = combined_data['W-L%']

    X_train, X_test, y_train, y_test = train_test_split(features.values, target.values, test_size=0.2, random_state=42)

    # Normalize
    mean = X_train.mean(axis=0)
    std  = X_train.std(axis=0)
    X_train = (X_train - mean) / std
    X_test  = (X_test  - mean) / std

    model = build_model(X_train.shape[1])
    model.fit(X_train, y_train, epochs=100, batch_size=8, verbose=0)

    # Save normalization params alongside the model
    return model, mean, std

In [ ]:
def save_model(NFL_YEAR, model, mean, std):
    os.makedirs(MODEL_FILEPATH, exist_ok=True)
    model.save(os.path.join(MODEL_FILEPATH, f"nn_model_{NFL_YEAR}.keras"))
    np.save(os.path.join(MODEL_FILEPATH, f"nn_mean_{NFL_YEAR}.npy"), mean)
    np.save(os.path.join(MODEL_FILEPATH, f"nn_std_{NFL_YEAR}.npy"),  std)

In [ ]:
def make_predictions(NFL_YEAR):
    model = keras.models.load_model(os.path.join(MODEL_FILEPATH, f"nn_model_{NFL_YEAR}.keras"))
    mean  = np.load(os.path.join(MODEL_FILEPATH, f"nn_mean_{NFL_YEAR}.npy"))
    std   = np.load(os.path.join(MODEL_FILEPATH, f"nn_std_{NFL_YEAR}.npy"))

    schedule_df = pd.read_csv(os.path.join("Schedule", f"{NFL_YEAR}.csv"))
    if schedule_df.empty:
        print(f"No schedule found for {NFL_YEAR}")
        return

    season_data = pd.read_csv(os.path.join(DATA_FILEPATH, f"{NFL_YEAR}.csv"))
    team_stats  = {row.iloc[TEAM]: read_data(NFL_YEAR, row.iloc[TEAM]) for _, row in season_data.iterrows()}
    team_stats  = {k: v for k, v in team_stats.items() if v is not None and len(v) > 0}

    team_wins  = {team: 0 for team in NFL_TEAMS}
    team_games = {team: 0 for team in NFL_TEAMS}

    for _, row in schedule_df.iterrows():
        winner, loser = row['Winner/tie'], row['Loser/tie']
        if winner not in team_stats or loser not in team_stats:
            continue

        def predict_team(team):
            X = team_stats[team].drop(columns=DROP_COLS).values
            X = (X - mean) / std
            return model.predict(X, verbose=0)[0][0]

        winner_pred = predict_team(winner)
        loser_pred  = predict_team(loser)

        predicted_winner = winner if winner_pred > loser_pred else loser
        predicted_loser  = loser  if winner_pred > loser_pred else winner
        team_wins[predicted_winner]  += 1
        team_games[predicted_winner] += 1
        team_games[predicted_loser]  += 1

    predictions = [
        [team,
         team_stats[team]['W-L%'].iloc[0] if team in team_stats else 0.0,
         team_wins[team] / team_games[team] if team_games[team] > 0 else 0.0]
        for team in NFL_TEAMS
    ]
    predictions_df = pd.DataFrame(predictions, columns=['Team', 'Actual W-L%', 'Predicted W-L%'])
    os.makedirs(PREDICTIONS_FILEPATH, exist_ok=True)
    predictions_df.to_csv(os.path.join(PREDICTIONS_FILEPATH, f"predictions_{NFL_YEAR}.csv"), index=False)
    return predictions_df

In [ ]:
for year in NFL_YEARS:
    print(f"Training neural network for {year}...")
    model, mean, std = train_model(year)
    save_model(year, model, mean, std)
    print(f"  Model saved.")
    preds = make_predictions(year)
    print(f"  Predictions made.\n")
    display(preds)

# Random Forest Model

## Simon Liedtke

# Naive Bayes Model

## Dilen Patel